# FLUPS 1: t0 calibration from dye runs

Template on 260605, the t0 runs `run01` and `run02` on PBBO (251 delays, -1.5 to 3.5 ps). No PCF
and no shift here: the t0 curve is determined on `wl_u`, the upconverted wavelength of the
device, and written as a calibration file for `flups_3_processing`.

Steps: read, despike, compare the scans, reject bad data, average, choose bands, kinetics of each
band, fit each band (t0 of the rise), fit the t0 curve, write it as a calibration file.

The runs are scans of the same t0 measurement: stacked, checked and averaged as in TA
(`ta_2_processing`). Which runs belong together is your choice: list them in `fnames`.

Each interactive tool is preceded by a **decisions** cell: a literal that the step after the tool
uses, so "Run all" gives the same result without clicking. Choices (scientific or taste) are
marked `# choice`.

Needs `%matplotlib widget` (JupyterLab or VS Code, with `ipympl`).

In [ ]:
%matplotlib widget
from math import ceil
from pathlib import Path

import lmfit
import matplotlib.pyplot as plt
import numpy as np
from lmfit.models import ConstantModel
from numpy.polynomial import Polynomial

from edyx.calib import write_calibration
from edyx.fana import despike_median
from edyx.io import read_dat, stack_traces
from edyx.models import conv_exp
from edyx.plot import plot_maps
from edyx.tana import avg_scans, band_average
from pyspan import Overlay, ScanInspector
from pyspan.decisions import rmsd_vs_median

## Parameters

In [ ]:
data_dir = Path("../../../edyx/examples/260605")  # edyx example data; in the lab: Path(".")
fnames = [data_dir / "260605_run01.dat", data_dir / "260605_run02.dat"]  # t0 runs, one dye
date = data_dir.resolve().name  # or enter a date manually
dye = "PBBO"

out_dir = Path(".")
t0_file = out_dir / f"{date}_t0_{dye}.toml"

despike_size = (5, 5)  # choice: median filter size (pixels, delays)
despike_threshold = 9  # counts. ~2 x (p95 - p5) of 250-262 nm (no signal), t < -0.8 ps: 4.3 -> 8.6
names = [f.name for f in fnames]
names

## Read and despike

Axes of `stack`: (file, wl_u, delay). The threshold is computed once on a background region and
pasted above. The check figure shows the raw map, the despiked map and what was removed. Above
360 nm there is only noise: spikes are replaced there too, which does not matter for the bands
used below.

In [ ]:
t, wl_u, stack = stack_traces([read_dat(f) for f in fnames])
despiked = np.array([despike_median(s, despike_size, threshold=despike_threshold) for s in stack])
stack.shape

In [ ]:
i = 0  # choice: scan shown in the check
fig, axes = plot_maps(t, wl_u, [stack[i], despiked[i], stack[i] - despiked[i]],
                      titles=["raw", "despiked", "removed"], ncols=3, vmin=-30, vmax=30)

## Compare the scans

In [ ]:
Overlay.from_stack(t, wl_u, despiked, labels=names, mode="kinetics").show()

## Reject data

RMSD of each scan against the median of the scans. With 2 files the median is the mean, so both
files get the same RMSD and the threshold cannot single out one file.

In [ ]:
rmsd = rmsd_vs_median(despiked, axis=0)

In [ ]:
# Paste the output of "Done" here. Format: (file, wl, t) indices, None = all,
# (start, stop) = range, stop excluded. E.g. [(1, None, (250, None))]
rejections = []
threshold = None  # None: no threshold

In [ ]:
insp = ScanInspector(t, wl_u, despiked, labels=names, rejections=rejections,
                     rmsd=rmsd, threshold=threshold)
insp.show()

In [ ]:
avg = avg_scans(insp.masked())

## Look at the data

Fluorescence of PBBO appears between 280 and 350 nm (`wl_u`), peak near 300 nm. The rise is
delayed at short wavelengths and earlier at long wavelengths (the t0 curve). Above 360 nm there
is no usable signal, only noise. No cross-correlation or Raman line is visible: below 275 nm
only a faint trace follows the fluorescence edge.

In [ ]:
fig, axes = plot_maps(t, wl_u, [avg], titles=["average"], vmin=-30, vmax=30)
axes[0, 0].set_ylim(250, 400)

## Bands

**Decisions**: bands `{center: (lo, hi)}` in nm of `wl_u`. Here 2 nm wide bands every 3 nm over
the fluorescence, 283-346 nm. The edge of the fluorescence at 280 nm and the noise above 350 nm
set the limits.

If the data has a cross-correlation or Raman line (a band that rises and falls within the
pulse), add it to the bands. Its kinetic is fitted with a Gaussian and the center is t0:

```python
model_cc = lmfit.models.GaussianModel() + ConstantModel()
r = model_cc.fit(k, params=model_cc.make_params(amplitude=100, center=0, sigma=0.1, c=0), x=t)
# t0 = r.params["center"].value, r.params["center"].stderr
```

With several dyes (to cover a wider range), process the runs of each dye up to the t0 points,
and stack the points of all dyes before the fit of the curve.

In [ ]:
bands = {lo + 1: (lo, lo + 2) for lo in range(282, 346, 3)}  # choice
kins = {c: np.ma.filled(band_average(wl_u, avg, b), np.nan) for c, b in bands.items()}

## Fit the dye bands

Plain lmfit, your model: `conv_exp` (exponential decay convolved with a gaussian, plus a step)
and a constant for the baseline before t0. `t0` of `conv_exp` is the time of the rise. The decay
is slow compared to the 5 ps window for long wavelengths: `tau` is then not determined, and the
fit reports no uncertainty or a huge one (first pass). Rejected points (NaN) are left out of the
fits.

In [ ]:
model = lmfit.Model(conv_exp) + ConstantModel()

def make_params(k, tau=None):
    p = model.make_params()
    p["amp"].set(value=5, min=0)  # choice: starting values and bounds
    p["tau"].set(value=5, min=0.02, max=500)
    p["sigma"].set(value=0.1, min=0.02, max=1)
    p["t0"].set(value=0)
    p["y0"].set(value=np.nanmean(k[-10:]))
    p["c"].set(value=np.nanmean(k[:20]), vary=False)  # baseline before t0
    if tau is not None:
        p["tau"].set(value=tau, vary=False)
    return p

first = {c: model.fit(k, params=make_params(k), x=t, nan_policy="omit") for c, k in kins.items()}

Second pass (optional): `tau` fixed to the weighted mean of the first-pass values (weights
`1/stderr**2`; bands without uncertainty are left out of the mean).

In [ ]:
taus = np.array([(r.params["tau"].value, r.params["tau"].stderr) for r in first.values()
                 if r.params["tau"].stderr is not None])
tau_mean = np.average(taus[:, 0], weights=taus[:, 1]**-2)
print(f"tau, weighted mean of {len(taus)} first-pass fits: {tau_mean:.3f} ps")

second = {c: model.fit(k, params=make_params(k, tau=tau_mean), x=t, nan_policy="omit")
          for c, k in kins.items()}

In [ ]:
results = second  # choice: first or second
ncols = 11
nrows = ceil(len(results) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2 * nrows), sharex=True, squeeze=False,
                         layout="constrained")
for ax in axes.flat[len(results):]:
    ax.set_visible(False)
for ax, (c, r) in zip(axes.flat, results.items()):
    ax.plot(t, kins[c], ".", ms=2)
    ax.plot(t[np.isfinite(kins[c])], r.best_fit, "k", lw=0.7)
    ax.axvline(r.params["t0"].value, color="r", lw=0.5)
    ax.set_title(f"{c} nm", fontsize="x-small")
    ax.set_xlim(-1, 1)

## t0 points

An array of rows `(wl_u, t0, t0_err)`. A band whose fit has no uncertainty (covariance not
estimated) is dropped, and printed.

In [ ]:
rows = []
for c, r in results.items():
    t0_c, err = r.params["t0"].value, r.params["t0"].stderr
    if err is None:
        print(f"dropped {c} nm: no uncertainty on t0")
        continue
    rows.append((c, t0_c, err))
points = np.array(rows)
points.shape

## Fit the t0 curve

`t0 = a0 + a1/wl_u**2 + a2/wl_u**4`: a polynomial of degree 2 in `1/wl_u**2`, weighted by
`1/t0_err`. The calibration file holds the sampled curve; the model and coefficients are kept for
information.

In [ ]:
pol = Polynomial.fit(points[:, 0]**-2, points[:, 1], 2, w=1/points[:, 2]).convert()  # choice
t0 = pol(wl_u**-2)
print("a0, a1, a2 =", pol.coef)

fig, ax = plt.subplots(figsize=(5, 3.5), layout="constrained")
ax.errorbar(points[:, 0], points[:, 1], points[:, 2], fmt=".", ms=4, capsize=2, label=dye)
sel = (wl_u > points[0, 0] - 5) & (wl_u < points[-1, 0] + 5)
ax.plot(wl_u[sel], t0[sel], "k", lw=0.7, label="fit")
ax.set_xlabel("wl_u / nm")
ax.set_ylabel("t0 / ps")
ax.legend()

In [ ]:
# Optional: reduced chi2 of the fit. Around 1 if the uncertainties describe the scatter.
resid = (points[:, 1] - pol(points[:, 0]**-2)) / points[:, 2]
print(f"reduced chi2: {(resid**2).sum() / (len(points) - 3):.2f}")

In [ ]:
fig, axes = plot_maps(t, wl_u, [avg], titles=["average, t0 points and curve"], vmin=-30, vmax=30)
axes[0, 0].plot(points[:, 1], points[:, 0], "k.", ms=3)
axes[0, 0].plot(t0, wl_u, color="k", lw=0.5)
axes[0, 0].set_xlim(-1.5, 1.5)
axes[0, 0].set_ylim(250, 400)

## Write the calibration file

On `wl_u`, the whole axis of the runs. Applied by value in `flups_3_processing`, where the
measurement's `wl_u` must lie within this range. Outside the range of the points the curve is
extrapolated.

In [ ]:
write_calibration(t0_file, "t0", "wl_u", wl_u, t0, units=["nm", "ps"], date=date,
                  source=f"{dye} {', '.join(names)}", made_by="flups_1_t0.ipynb",
                  fit={"model": "a0 + a1/wl_u**2 + a2/wl_u**4, low order first",
                       "coef": pol.coef, "points": points})
t0_file